In [1]:
import numpy as np

# -----------------------------
# Grid World Definition
# -----------------------------
n = 4
GOAL = (3, 3)
OBSTACLES = [(1, 1)]
ACTIONS = ['U', 'D', 'L', 'R']

# -----------------------------
# Reward Function
# -----------------------------
def reward(state):
    if state == GOAL:
        return 10
    elif state in OBSTACLES:
        return -5
    else:
        return -1

# -----------------------------
# Transition Function
# -----------------------------
def next_state(state, action):
    x, y = state

    if state == GOAL:
        return state

    if action == 'U':
        x = max(0, x - 1)
    elif action == 'D':
        x = min(n - 1, x + 1)
    elif action == 'L':
        y = max(0, y - 1)
    elif action == 'R':
        y = min(n - 1, y + 1)

    return (x, y)

# =====================================================
# PART A: VALUE ITERATION
# =====================================================
gamma = 0.9
theta = 1e-4

V = np.zeros((n, n))

def value_iteration():
    global V
    while True:
        delta = 0
        new_V = V.copy()

        for x in range(n):
            for y in range(n):
                state = (x, y)

                if state == GOAL:
                    continue

                values = []
                for a in ACTIONS:
                    s_next = next_state(state, a)
                    r = reward(s_next)
                    values.append(r + gamma * V[s_next])

                new_V[state] = max(values)
                delta = max(delta, abs(V[state] - new_V[state]))

        V = new_V
        if delta < theta:
            break

value_iteration()

# Extract Value Iteration Policy
policy_VI = np.full((n, n), '', dtype=str)

for x in range(n):
    for y in range(n):
        state = (x, y)

        if state == GOAL:
            policy_VI[state] = 'G'
            continue

        values = {}
        for a in ACTIONS:
            s_next = next_state(state, a)
            r = reward(s_next)
            values[a] = r + gamma * V[s_next]

        policy_VI[state] = max(values, key=values.get)

# =====================================================
# PART B: Q-LEARNING
# =====================================================
Q = np.zeros((n, n, len(ACTIONS)))

alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 500

for ep in range(episodes):
    state = (0, 0)

    while state != GOAL:
        x, y = state

        if np.random.rand() < epsilon:
            a_idx = np.random.randint(4)
        else:
            a_idx = np.argmax(Q[x, y])

        action = ACTIONS[a_idx]
        s_next = next_state(state, action)
        r = reward(s_next)
        nx, ny = s_next

        Q[x, y, a_idx] += alpha * (
            r + gamma * np.max(Q[nx, ny]) - Q[x, y, a_idx]
        )

        state = s_next

# Extract Q-Learning Policy
policy_QL = np.full((n, n), '', dtype=str)

for x in range(n):
    for y in range(n):
        if (x, y) == GOAL:
            policy_QL[x, y] = 'G'
        else:
            policy_QL[x, y] = ACTIONS[np.argmax(Q[x, y])]

# -----------------------------
# Print Policies
# -----------------------------
print("Value Iteration Policy:")
print(policy_VI)

print("\nQ-Learning Policy:")
print(policy_QL)

Value Iteration Policy:
[['D' 'R' 'D' 'D']
 ['D' 'D' 'D' 'D']
 ['D' 'D' 'D' 'D']
 ['R' 'R' 'R' 'G']]

Q-Learning Policy:
[['R' 'R' 'R' 'D']
 ['U' 'R' 'R' 'D']
 ['R' 'R' 'D' 'D']
 ['R' 'R' 'R' 'G']]
